In [1]:
# Test script 2

In [6]:
import os
import re
import glob
import cftime
import warnings
import xarray as xr
from collections import defaultdict
from utils.utils import get_scenario_config, load_file_list

In [7]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/file_paths/"
SCRATCH = f"/glade/derecho/scratch/awells/air_quality/{model}/pm25/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/annual_pm25/"

VAR_list = ["BC", "POA", "SOA", "SO4", "SS", "DU"]

In [8]:
def load_file(f):
    if not os.path.exists(f):
        raise ValueError(f"Missing: {f}")

    # Find variable
    pattern = re.compile(r"cam\.h0\.([^.]+)\.")
    m = pattern.search(f)
    varname = m.group(1)

    ds = xr.open_dataset(f)
    da = ds[varname]
    return da


def select_surface(da):
    return da.isel(lev=-1)


# CESM2 naming convention shifts months by 1
# the data represents 2015-01 - 2020-12 but time coord shows 2015-02 - 2021-01
def minus_one_month(date):
    """Subtract one month from a cftime.DatetimeNoLeap object."""
    year, month = date.year, date.month
    if month == 1:
        return cftime.DatetimeNoLeap(year - 1, 12, date.day,
                                     date.hour, date.minute, date.second,
                                     date.microsecond, has_year_zero=date.has_year_zero)
    else:
        return cftime.DatetimeNoLeap(year, month - 1, date.day,
                                     date.hour, date.minute, date.second,
                                     date.microsecond, has_year_zero=date.has_year_zero)

In [11]:
for var in ["T"]:
    for ens_num in [1]:
        print(f"Processing {var}, {scenario} ensemble {ens_num:02d}")
        # Load all file lists
        file_list = load_file_list(FILE_DIR, f"file_list_{var}_{scenario}_{ens_num:02d}.json")

        groups = defaultdict(list)
        for f in file_list:
            # Load surface variable
            da = select_surface(load_file(f))
            groups[da.name].append(da)

        combined = {}
        for vbl, das in groups.items():
            combined[vbl] = xr.concat(
                das,
                dim="time",
                join="outer",
                combine_attrs="drop_conflicts"
            )

        # Get unit attribute from combined
        first_key = next(iter(combined))
        units = combined[first_key].attrs["units"]

        total_var = sum(combined.values())
        total_var.attrs["units"] = units

        # If the first month is February (2) then apply month fixer
        first_month = total_var.time.dt.month[0]
        if first_month == 2:
            print("Adjusting month indexing")
            new_time = [minus_one_month(t) for t in total_var["time"].values]
            total_var = total_var.assign_coords(time=new_time)
        # If the first month is January (1) don't apply month fixer
        elif first_month == 1:
            print("No month index adjusting needed")
        else:
            warnings.warn(f"First month: {first_month}, check dates in file")

        first_year = total_var.time.dt.year[0].item()
        last_year = total_var.time.dt.year[-1].item()

        out_file = f"{var}_mmr_{model}_{scenario}_{ens_num:02d}_{first_year}-{last_year}.nc"
        out_path = os.path.join(SCRATCH, out_file)
        #total_var.to_netcdf(out_path)

print("All processing complete.")

Processing T, SSP245_G6 ensemble 01
Adjusting month indexing
All processing complete.


In [17]:
combined

{'T': <xarray.DataArray 'T' (time: 1032, lat: 192, lon: 288)> Size: 228MB
 array([[[247.70624, 247.70624, 247.70624, ..., 247.70624, 247.70624,
          247.70624],
         [248.33849, 248.3156 , 248.29239, ..., 248.41037, 248.38652,
          248.36174],
         [248.82292, 248.77243, 248.72073, ..., 248.97557, 248.92395,
          248.8721 ],
         ...,
         [243.46114, 243.50764, 243.56271, ..., 243.36671, 243.3933 ,
          243.42455],
         [244.4734 , 244.49416, 244.51862, ..., 244.42049, 244.43643,
          244.45485],
         [245.72127, 245.7213 , 245.7213 , ..., 245.72122, 245.72124,
          245.72128]],
 
        [[237.38408, 237.38406, 237.38406, ..., 237.38399, 237.38393,
          237.38405],
         [238.26543, 238.24763, 238.23087, ..., 238.31792, 238.29987,
          238.28119],
         [239.03561, 238.99443, 238.94977, ..., 239.14018, 239.10814,
          239.07132],
 ...
         [263.28265, 263.26428, 263.24646, ..., 263.31573, 263.3091 ,
      

In [14]:
# Load BC
bc_file = f"BC_mmr_{model}_{scenario}_{ens_num:02d}_*.nc"
bc_path = glob.glob(os.path.join(SCRATCH, bc_file))[0]
bc = xr.open_dataarray(bc_path)

In [15]:
import xarray as xr

q = bc  # kg/kg
p_hPa = total_var.lev  # pressure in hPa
T = total_var  # temperature in K

# convert pressure to Pa
p = p_hPa * 100.0

# compute density
Rd = 287.0  # J/kg/K
rho = p / (Rd * T)

# convert mixing ratio to μg/m3
BC_ugm3 = q * rho * 1e9

In [18]:
T

<xarray.DataArray 'T' (time: 1032, lat: 192, lon: 288)> Size: 228MB
array([[[247.70624, 247.70624, 247.70624, ..., 247.70624, 247.70624,
         247.70624],
        [248.33849, 248.3156 , 248.29239, ..., 248.41037, 248.38652,
         248.36174],
        [248.82292, 248.77243, 248.72073, ..., 248.97557, 248.92395,
         248.8721 ],
        ...,
        [243.46114, 243.50764, 243.56271, ..., 243.36671, 243.3933 ,
         243.42455],
        [244.4734 , 244.49416, 244.51862, ..., 244.42049, 244.43643,
         244.45485],
        [245.72127, 245.7213 , 245.7213 , ..., 245.72122, 245.72124,
         245.72128]],

       [[237.38408, 237.38406, 237.38406, ..., 237.38399, 237.38393,
         237.38405],
        [238.26543, 238.24763, 238.23087, ..., 238.31792, 238.29987,
         238.28119],
        [239.03561, 238.99443, 238.94977, ..., 239.14018, 239.10814,
         239.07132],
...
        [263.28265, 263.26428, 263.24646, ..., 263.31573, 263.3091 ,
         263.29752],
        [263.1284 , 263.1175 , 263.10678, ..., 263.1581 , 263.14896,
         263.13892],
        [262.92767, 262.92776, 262.92783, ..., 262.9274 , 262.92752,
         262.92758]],

       [[252.65015, 252.65015, 252.65015, ..., 252.65015, 252.65015,
         252.65015],
        [253.45914, 253.43729, 253.41743, ..., 253.52672, 253.50406,
         253.48131],
        [254.061  , 254.00693, 253.95105, ..., 254.22194, 254.16747,
         254.11366],
        ...,
        [254.89679, 254.92421, 254.9489 , ..., 254.7913 , 254.83243,
         254.86688],
        [254.52852, 254.55397, 254.57834, ..., 254.45349, 254.47772,
         254.50293],
        [253.97289, 253.9729 , 253.9729 , ..., 253.97285, 253.97287,
         253.97289]]], dtype=float32)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
    lev      float64 8B 992.6
  * time     (time) object 8kB 2015-01-01 00:00:00 ... 2100-12-01 00:00:00
Attributes:
    units:    K